# 📈 Sales Prediction — Exploratory Analysis & Model Prototyping

This notebook explores the advertising dataset, performs EDA, and prototypes the regression models used in the main project (`train_model.py` / `app.py`).

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_style('whitegrid')
%matplotlib inline

## 1. Load Dataset

In [ ]:
df = pd.read_csv('../dataset/advertising.csv')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 2. Data Cleaning

In [ ]:
print('Missing values:\n', df.isnull().sum())
print('\nDuplicate rows:', df.duplicated().sum())

df = df.drop_duplicates()
df = df.dropna()
print('\nShape after cleaning:', df.shape)

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, col in zip(axes, df.columns):
    sns.histplot(df[col], kde=True, ax=ax, color='#FF4B4B')
    ax.set_title(f'Distribution of {col}')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ['TV', 'Radio', 'Newspaper']):
    sns.scatterplot(x=df[col], y=df['Sales'], ax=ax, color='#F7931E')
    ax.set_title(f'{col} vs Sales')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(df.corr(), annot=True, cmap='Reds', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
sns.pairplot(df, diag_kind='kde', corner=True)
plt.show()

## 4. Train / Test Split & Feature Scaling

In [ ]:
X = df[['TV', 'Radio', 'Newspaper']]
y = df['Sales']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Train shape:', X_train.shape, '| Test shape:', X_test.shape)

## 5. Train & Compare Models

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree Regressor': DecisionTreeRegressor(random_state=42),
    'Random Forest Regressor': RandomForestRegressor(n_estimators=200, random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    results[name] = {
        'MAE': mean_absolute_error(y_test, y_pred),
        'MSE': mean_squared_error(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'R2 Score': r2_score(y_test, y_pred),
    }

results_df = pd.DataFrame(results).T
results_df.sort_values('R2 Score', ascending=False)

## 6. Best Model & Feature Importance

In [ ]:
best_model_name = results_df['R2 Score'].idxmax()
print('Best model:', best_model_name)

best_model = models[best_model_name]
if hasattr(best_model, 'feature_importances_'):
    importance_df = pd.DataFrame({
        'Feature': X.columns,
        'Importance': best_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    display(importance_df)
else:
    coef_df = pd.DataFrame({
        'Feature': X.columns,
        'Coefficient': best_model.coef_
    })
    display(coef_df)

## 7. Predicted vs Actual & Residuals

In [ ]:
y_pred_best = best_model.predict(X_test_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, y_pred_best, color='#FF4B4B', alpha=0.7)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--')
axes[0].set_xlabel('Actual Sales')
axes[0].set_ylabel('Predicted Sales')
axes[0].set_title('Predicted vs Actual')

residuals = y_test - y_pred_best
axes[1].scatter(y_pred_best, residuals, color='#150458', alpha=0.7)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel('Predicted Sales')
axes[1].set_ylabel('Residuals')
axes[1].set_title('Residual Plot')

plt.tight_layout()
plt.show()

## 8. Sample Prediction

In [ ]:
sample = pd.DataFrame([[200, 35, 50]], columns=['TV', 'Radio', 'Newspaper'])
sample_scaled = scaler.transform(sample)
predicted_sales = best_model.predict(sample_scaled)[0]
print(f'Predicted Sales for TV=200, Radio=35, Newspaper=50: {predicted_sales:.2f} Units')

---
This notebook is for exploration/prototyping only. The production pipeline (used by the Streamlit app) lives in `train_model.py`, `predict.py`, `utils.py`, and `config.py` at the project root.